# Seed the `watchlist` table with synthetic daily scores

Populates `ml.watchlist` with realistic **daily OneGrid scores through today** for demo purposes (stand-in for the real modeling notebooks).

- Learns one **template per signature** `(model_name, asset_id, tag_name, feature)` from the existing watchlist — preserving tags, descriptors, baselines and horizons.
- Generates a score per template per day for the last `DAYS` days, with a mild escalation trend so a few items rise to **HIGH/CRITICAL** near today.
- `recommended_action` is derived per model on its native scale (AAKR/Anomaly z-scores; GBM stop-probability; Cox long-term).
- **Idempotent**: with `PRESERVE_HISTORY=True` it keeps original rows older than the window and replaces the window each run.

Run all. Seeded rows carry `notebook_run_id` starting `demo-seed:`.

In [ ]:
# PARAMETERS  (Fabric: this cell is tagged 'parameters')
WORKSPACE_ID = "163ba38c-3869-406f-adb7-37cbc981390c"
LAKEHOUSE_ID = "7e08480c-cf8d-4206-901d-38b74dbe35d9"           # lh_poc
TABLE_PATH   = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/ml/watchlist"
NARRATIVE_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/ml/daily_narrative"
DAYS         = 14        # generate daily scores for the last N days ending today (inclusive)
SEED         = 7
PRESERVE_HISTORY = True  # keep original rows older than the window; replace the window each run (idempotent)
RUN_TAG      = "demo-seed"
MAX_CRITICAL_TAGS = 5    # cap distinct CRITICAL tags per day for demo clarity (< 6); extras demoted to HIGH
ROOTCAUSE_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables/ml/root_cause"


In [ ]:
# Read the existing watchlist to learn realistic templates + exact schema.
from pyspark.sql import functions as F
existing = spark.read.format("delta").load(TABLE_PATH)
schema = existing.schema
cols = existing.columns
print("existing rows:", existing.count(), "| columns:", len(cols))
pdf_all = existing.toPandas()


In [ ]:
# One template per (model_name, asset_id, tag_name, feature) = the latest scored row for that signature.
import pandas as pd
pdf_all["_sd"] = pdf_all["scoring_date"].astype(str)
key = ["model_name", "asset_id", "tag_name", "feature"]
templates = (pdf_all.sort_values("_sd")
             .groupby(key, dropna=False, as_index=False)
             .tail(1)
             .drop(columns=["_sd"])
             .reset_index(drop=True))
print("templates:", len(templates))
templates[["model_name","asset_id","tag_name","feature","recommended_action","risk_contribution"]].head(8)


In [ ]:
# Generate fresh daily scores through today for every template.
import numpy as np, pandas as pd, uuid, datetime as dt

rng = np.random.default_rng(SEED)
today = dt.datetime.now(dt.timezone.utc).date()
days  = [today - dt.timedelta(days=i) for i in range(DAYS - 1, -1, -1)]
window_start = min(days).isoformat()
run_id = f"{RUN_TAG}:{uuid.uuid4()}"

def action_from(model, risk, stop_prob):
    if model in ("AAKR_SmartSignal", "AnomalyDetection_SmartSignal"):
        z = risk if risk is not None else 0.0
        return "CRITICAL" if z >= 50 else "HIGH" if z >= 20 else "MEDIUM" if z >= 8 else "LOW"
    if model == "GBM_ShortTerm":
        p = stop_prob
        return "CRITICAL" if p >= 0.7 else "HIGH" if p >= 0.5 else "MEDIUM" if p >= 0.3 else "LOW"
    # CoxPH_LongTerm
    return "HIGH" if (risk or 0) >= 0.2 else "MEDIUM" if (risk or 0) >= 0.02 else "LOW"

def fnum(x, default):
    try:
        return float(x) if x is not None and not pd.isna(x) else default
    except Exception:
        return default

rows = []
for _, tmpl in templates.iterrows():
    model = tmpl["model_name"]
    base_risk = fnum(tmpl.get("risk_contribution"), None)
    # a mild trend so a few items escalate toward "today" (more interesting demo)
    trend = rng.uniform(0.9, 1.25)
    for di, d in enumerate(days):
        frac = (di + 1) / len(days)
        # per-model risk / probability generation
        if model in ("AAKR_SmartSignal", "AnomalyDetection_SmartSignal"):
            b = base_risk if (base_risk and base_risk > 0) else rng.uniform(8, 60)
            risk = max(0.1, b * (trend ** frac) * float(np.exp(rng.normal(0, 0.25))))
            stop_prob = None
        elif model == "GBM_ShortTerm":
            risk = max(0.001, fnum(tmpl.get("risk_contribution"), 0.02) * float(np.exp(rng.normal(0, 0.3))))
            stop_prob = float(np.clip(0.45 * (trend ** frac) + rng.normal(0, 0.08), 0.02, 0.95))
        else:  # CoxPH_LongTerm
            risk = float(np.clip(fnum(tmpl.get("risk_contribution"), 0.02) * (trend ** frac)
                                 * float(np.exp(rng.normal(0, 0.2))), 0.0005, 0.6))
            stop_prob = None
        action = action_from(model, risk, stop_prob)

        bmean = fnum(tmpl.get("baseline_mean"), 0.0)
        bstd  = fnum(tmpl.get("baseline_std"), 1.0) or 1.0
        cur   = fnum(tmpl.get("current_value"), bmean)
        cur   = cur + rng.normal(0, abs(bstd) * 0.05 + 1e-6)

        r = tmpl.to_dict()
        r["scoring_date"] = d.isoformat()
        r["recommended_action"] = action
        r["recommended_action_orig"] = action
        r["risk_contribution"] = risk
        r["current_value"] = cur
        r["model_run_timestamp"] = dt.datetime(d.year, d.month, d.day, 11, 35, 0, tzinfo=dt.timezone.utc)
        r["notebook_run_id"] = run_id
        if model == "GBM_ShortTerm" and stop_prob is not None:
            r["driver_value"] = float(stop_prob)
        rows.append(r)

gen = pd.DataFrame(rows)[templates.columns.tolist() if False else cols]

# Cap distinct CRITICAL tags per day so the demo stays focused (< 6). Extras demoted to HIGH.
gen["scoring_date"] = gen["scoring_date"].astype(str)
_cap = int(MAX_CRITICAL_TAGS) if str(MAX_CRITICAL_TAGS).strip() else 0
if _cap and _cap > 0:
    def _cap_day(g):
        c = g[(g["recommended_action"] == "CRITICAL") & g["tag_name"].notna()
              & (~g["tag_name"].astype(str).str.contains("__", na=False))]
        keep = set(c.groupby("tag_name")["risk_contribution"].max()
                   .sort_values(ascending=False).head(_cap).index)
        dm = (g["recommended_action"] == "CRITICAL") & (~g["tag_name"].isin(keep))
        g.loc[dm, "recommended_action"] = "HIGH"
        return g
    gen = gen.groupby("scoring_date", group_keys=False).apply(_cap_day)
    print(f"Capped CRITICAL to <= {_cap} distinct tags per day.")

print(f"Generated {len(gen):,} rows over {len(days)} days ({window_start} -> {today}); run_id={run_id}")
print("action mix (today):")
print(gen[gen["scoring_date"] == today.isoformat()]["recommended_action"].value_counts().to_dict())


In [ ]:
# Write to the lakehouse watchlist Delta table.
#  - PRESERVE_HISTORY: keep original rows OLDER than the window, replace the window (idempotent re-runs).
from pyspark.sql import functions as F

gen_sdf = spark.createDataFrame(gen[cols], schema=schema)

if PRESERVE_HISTORY:
    kept = existing.where(F.col("scoring_date") < F.lit(window_start))
    out = kept.unionByName(gen_sdf)
else:
    out = gen_sdf

(out.write.format("delta").mode("overwrite").option("overwriteSchema", "false").save(TABLE_PATH))
print("Wrote watchlist. Total rows now:", spark.read.format("delta").load(TABLE_PATH).count())


In [ ]:
# Root-cause correlation for today's CRITICAL tags -> ml.root_cause (for the chat agent to explain).
import pandas as pd, datetime as dt
from pyspark.sql import types as T

today_str = dt.datetime.now(dt.timezone.utc).date().isoformat()
tdf = gen[gen["scoring_date"] == today_str].copy()
crit = tdf[(tdf["recommended_action"] == "CRITICAL") & tdf["tag_name"].notna()
           & (~tdf["tag_name"].astype(str).str.contains("__", na=False))]
crit = crit.sort_values("risk_contribution", ascending=False).groupby("tag_name", as_index=False).first()

def mechanism(desc):
    d0 = (desc or "").lower()
    if any(k in d0 for k in ["lvl", "level", "valve", "vv"]):
        return ("Level-control valve hunting",
                "A control valve is not tracking its demand, so the measured feedback swings outside its normal band; an upstream flow or condensate imbalance is the likely driver.",
                "Inspect the level-control valve and positioner; check condensate/feedwater flow balance and valve calibration.")
    if any(k in d0 for k in ["speed", "rpm", "turb"]):
        return ("Governor / rotational instability",
                "Rotational speed is deviating from setpoint, consistent with governor-valve response lag or rising mechanical drag on the shaft.",
                "Check governor-valve response and lube-oil supply; review bearing temperatures and vibration for drag.")
    if any(k in d0 for k in ["press", "drum", "backpress"]):
        return ("Flow / combustion imbalance",
                "Pressure is trending off-baseline, consistent with a feedwater-versus-steam-flow imbalance or a combustion swing.",
                "Rebalance feedwater/steam flow; verify drum-level control and combustion tuning.")
    if any(k in d0 for k in ["temp", "therm"]):
        return ("Reduced cooling / heat-transfer",
                "Temperature is climbing above baseline, consistent with reduced cooling flow or heat-exchanger fouling.",
                "Verify cooling-water flow and heat-exchanger cleanliness; check for fouling or a flow restriction.")
    if any(k in d0 for k in ["vib", "vibr"]):
        return ("Mechanical / bearing wear",
                "Vibration is elevated above baseline, consistent with bearing wear, imbalance, or misalignment.",
                "Trend bearing temperatures and vibration; schedule an alignment/bearing inspection.")
    if "flow" in d0:
        return ("Pump degradation / restriction",
                "Flow is deviating from baseline, consistent with pump wear or a downstream restriction.",
                "Check pump discharge pressure and the strainer/valve line-up for a restriction.")
    return ("Process deviation",
            "The signal has drifted outside its normal operating band; correlated process signals indicate an emerging upset.",
            "Investigate the correlated signals and confirm control-loop performance.")

def fnum(x):
    try:
        return float(x)
    except Exception:
        return None

recs = []
gen_ts = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
for r in crit.itertuples(index=False):
    peers = tdf[(tdf["asset_id"] == r.asset_id) & (tdf["tag_name"] != r.tag_name) & tdf["tag_name"].notna()
                & (~tdf["tag_name"].astype(str).str.contains("__", na=False))].copy()
    peers["_pri"] = peers["recommended_action"].map({"CRITICAL": 3, "HIGH": 2, "MEDIUM": 1, "LOW": 0}).fillna(0)
    peers = peers.sort_values(["_pri", "risk_contribution"], ascending=False).drop_duplicates("tag_name").head(3)
    mech, narr, action = mechanism(getattr(r, "descriptor", ""))
    contrib_names = [str(t) for t in peers["tag_name"].tolist()]
    if len(peers):
        contrib_desc = "; ".join(f"{p.tag_name} ({p.descriptor}, {p.trend_direction})" for p in peers.itertuples(index=False))
    else:
        contrib_desc = "correlated process signals on the same asset"
    cv = fnum(r.current_value)
    trend = str(getattr(r, "trend_direction", "") or "").lower()
    tw = f", trending {trend}" if trend and trend not in ("stable", "flat", "none", "") else ""
    dev = f" It is currently reading {cv:.1f}{tw}, outside its normal operating band." if cv is not None else ""
    full = f"{narr}{dev} Contributing signals: {contrib_desc}. Recommended action: {action}"
    conf = round(min(0.95, 0.72 + 0.06 * min(len(peers), 3)), 2)
    recs.append(dict(scoring_date=today_str, asset_id=str(r.asset_id), tag=str(r.tag_name),
                     descriptor=str(getattr(r, "descriptor", "") or ""), priority="CRITICAL",
                     root_cause=full, failure_mechanism=mech, contributing_tags=contrib_desc,
                     contributing_tag_names=";".join(contrib_names),
                     recommended_action=action, confidence=float(conf), generated_at=gen_ts))

rc_cols = ["scoring_date", "asset_id", "tag", "descriptor", "priority", "root_cause", "failure_mechanism",
           "contributing_tags", "contributing_tag_names", "recommended_action", "confidence", "generated_at"]
rc_schema = T.StructType([T.StructField(c, T.DoubleType() if c == "confidence" else T.StringType(), True) for c in rc_cols])
rc_sdf = spark.createDataFrame(pd.DataFrame(recs, columns=rc_cols), schema=rc_schema)
rc_sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(ROOTCAUSE_PATH)
print(f"Wrote ml.root_cause: {len(recs)} critical root-cause record(s) for {today_str}.")
for rec in recs:
    print("  -", rec["tag"], "->", rec["failure_mechanism"])


In [ ]:
# Verify: recent-day counts + action mix for today (what the chat agent / report will show).
import datetime as dt
today = dt.datetime.now(dt.timezone.utc).date().isoformat()
wl = spark.read.format("delta").load(TABLE_PATH)
print("Rows scored TODAY (%s): %d" % (today, wl.where(F.col("scoring_date") == today).count()))
(wl.where(F.col("scoring_date") == today)
   .groupBy("recommended_action").count().orderBy(F.desc("count")).show())
(wl.where(F.col("scoring_date") == today)
   .select("asset_id", "model_name", "tag_name", "recommended_action", F.round("risk_contribution", 3).alias("risk"))
   .orderBy(F.desc("recommended_action")).show(20, truncate=False))


In [ ]:
# Generate the daily NARRATIVE summary from today's watchlist and OVERWRITE ml.daily_narrative (single row).
import datetime as dt, html as _html
today_str = dt.datetime.now(dt.timezone.utc).date().isoformat()
wl = spark.read.format("delta").load(TABLE_PATH).where(F.col("scoring_date") == F.lit(today_str)).toPandas()

def friendly(asset_id):
    p = str(asset_id).split("_")
    if len(p) >= 2 and p[1][:1].upper() == "U" and p[1][1:].isdigit():
        return (p[0] + " Unit " + p[1][1:] + " " + " ".join(w.capitalize() for w in p[2:])).strip()
    return str(asset_id).replace("_", " ")

ORDER = {"CRITICAL": 3, "HIGH": 2, "MEDIUM": 1, "LOW": 0}
wl["_rank"] = wl["recommended_action"].map(ORDER).fillna(0)
crit = wl[wl["recommended_action"] == "CRITICAL"]
high = wl[wl["recommended_action"] == "HIGH"]
actionable = wl[wl["recommended_action"].isin(["CRITICAL", "HIGH"])]

lines, html_rows = [], []
for asset, grp in actionable.sort_values("_rank", ascending=False).groupby("asset_id", sort=False):
    status = "CRITICAL" if (grp["recommended_action"] == "CRITICAL").any() else "HIGH"
    descs = [d for d in grp["descriptor"].dropna().unique().tolist() if d][:5]
    models = "+".join(sorted(grp["model_name"].dropna().unique().tolist()))
    worst = float(grp["risk_contribution"].max()) if grp["risk_contribution"].notna().any() else 0.0
    watch = ", ".join(descs) if descs else "flagged sensors"
    lines.append(f"[{status}] {friendly(asset)}: Watch {watch}. Worst risk={worst:.1f}. Sources: {models}.")
    color = "#dc3545" if status == "CRITICAL" else "#fd7e14"
    html_rows.append(
        f"<tr><td style='padding:4px;border-bottom:1px solid #eee'><strong style='color:{color}'>{status}</strong></td>"
        f"<td style='padding:4px;border-bottom:1px solid #eee'>{_html.escape(friendly(asset))}</td>"
        f"<td style='padding:4px;border-bottom:1px solid #eee'>{_html.escape(watch)}</td>"
        f"<td style='padding:4px;border-bottom:1px solid #eee'>{_html.escape(models)}</td></tr>")

if len(crit): system_status = "CRITICAL"
elif len(high): system_status = "HIGH"
elif (wl["recommended_action"] == "MEDIUM").any(): system_status = "MEDIUM"
else: system_status = "NORMAL"

narrative_text = " | ".join(lines) if lines else f"[NORMAL] No CRITICAL/HIGH alerts for {today_str}. All monitored assets within normal operating bands."
narrative_html = ("<table style=\"font-family:'Segoe UI',Arial,sans-serif;width:100%;border-collapse:collapse;font-size:11px;line-height:1.3\">"
                  "<tr><td colspan='4' style='padding:4px;border-bottom:2px solid #ccc;background:#f8f9fa'>"
                  f"<strong>Daily watch summary &mdash; {today_str}</strong> &nbsp; "
                  f"<span style='color:#dc3545'>{len(crit)} critical</span>, {len(actionable)} actionable, "
                  f"{actionable['asset_id'].nunique()} assets flagged</td></tr>"
                  + "".join(html_rows) + "</table>")

narr = {
    "narrative_date": today_str,
    "narrative_text": narrative_text,
    "narrative_html": narrative_html,
    "system_status": system_status,
    "critical_alerts": int(len(crit)),
    "total_alerts": int(len(actionable)),
    "assets_flagged": int(actionable["asset_id"].nunique()),
    "generated_at": dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%S"),
}
print("system_status=%s  critical=%d  actionable=%d  assets=%d"
      % (system_status, len(crit), len(actionable), narr["assets_flagged"]))
print("narrative_text:", narrative_text[:400])

# Overwrite the single-row narrative table (schema taken from the existing table).
narr_schema = spark.read.format("delta").load(NARRATIVE_PATH).schema
import pandas as pd
narr_sdf = spark.createDataFrame(pd.DataFrame([narr])[[f.name for f in narr_schema]], schema=narr_schema)
narr_sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "false").save(NARRATIVE_PATH)
print("daily_narrative overwritten for", today_str)


In [ ]:
# Refresh the Critical-Tag-Monitor "Today's Critical Tags" spotlight from today's watchlist (best-effort).
try:
    import requests, base64 as _b64, notebookutils
    API = "https://api.fabric.microsoft.com/v1"
    ftok = None
    for _aud in ("https://api.fabric.microsoft.com", "pbi", "fabric"):
        try:
            ftok = notebookutils.credentials.getToken(_aud)
            if ftok: break
        except Exception:
            pass
    HDR = {"Authorization": f"Bearer {ftok}", "Content-Type": "application/json"}
    PAGE_ID, Q_TABLE, Q_LINE, T_TABLE, T_LINE = (
        "c0ffee00-0000-0000-0000-000000000001", "c0ffee00-0000-0000-0000-000000000002",
        "c0ffee00-0000-0000-0000-000000000003", "c0ffee00-0000-0000-0000-000000000004",
        "c0ffee00-0000-0000-0000-000000000005")

    def _lro(resp):
        if resp.status_code in (200, 201):
            return resp.json() if resp.text else {}
        if resp.status_code == 202:
            loc = resp.headers.get("Location")
            for _ in range(60):
                import time as _t; _t.sleep(3)
                s = requests.get(loc, headers=HDR)
                st = (s.json() or {}).get("status")
                if st in ("Succeeded", "Completed"):
                    r = requests.get(loc.rstrip("/") + "/result", headers=HDR)
                    return r.json() if r.text else {}
                if st == "Failed":
                    raise RuntimeError("LRO failed")
        resp.raise_for_status()
        return {}

    # today's CRITICAL raw tags from the watchlist just written
    wl_today = spark.read.format("delta").load(TABLE_PATH).where(F.col("scoring_date") == F.lit(today_str)).toPandas()
    crit = wl_today[(wl_today["recommended_action"] == "CRITICAL") & wl_today["tag_name"].notna()
                    & (~wl_today["tag_name"].astype(str).str.contains("__", na=False))].copy()
    crit = (crit.sort_values("risk_contribution", ascending=False)
                .groupby(["tag_name", "descriptor"], as_index=False).first()
                .sort_values("risk_contribution", ascending=False).head(12))
    tags = [(r.tag_name, (r.descriptor or ""), float(r.risk_contribution)) for r in crit.itertuples(index=False)]
    if not tags:
        raise RuntimeError("no critical tags today")

    def _esc(s):
        return str(s).replace("\\", "\\\\").replace('"', '\\"')
    drows = ",\n    ".join(f'"{_esc(t)}","{_esc(d0)}",{rk:.4f}' for t, d0, rk in tags)
    # Expected + Std come from each tag's OWN PiEvents history (same scale as live Actual) -> meaningful Sigma.
    table_text = ("let crit = datatable(Tag:string, Descriptor:string, Risk:real)[\n    " + drows + "\n];\n"
                  "let baseline = PiEvents\n| where Tag in (crit | project Tag) and not(Questionable) and Ts between (ago(14d) .. ago(1h))\n"
                  "| summarize Expected=avg(toreal(Value)), Std=stdev(toreal(Value)) by Tag;\n"
                  "PiEvents\n| where Tag in (crit | project Tag) and not(Questionable) and IngestedAt > ago(10m)\n"
                  "| summarize arg_max(Ts, Value) by Tag\n| join kind=inner crit on Tag\n| join kind=leftouter baseline on Tag\n"
                  "| extend Actual=round(toreal(Value),2), Expected=round(Expected,2)\n"
                  "| extend Deviation=round(Actual-Expected,2), Sigma=iff(isnull(Std) or Std<=0,real(null),round((Actual-Expected)/Std,1))\n"
                  "| project Priority=\"CRITICAL\", Descriptor, Tag, Actual, Expected, Deviation, Sigma, RiskZ=round(Risk,1), Updated=Ts\n"
                  "| order by RiskZ desc")
    top = [t for t, _, _ in tags[:8]]
    line_text = ("let crit = dynamic([" + ",".join(f'"{_esc(t)}"' for t in top) + "]);\n"
                 "PiEvents\n| where Tag in (crit) and Ts > ago(1h)\n"
                 "| summarize Value=avg(toreal(Value)) by bin(Ts, 1m), Tag\n| order by Ts asc")

    items = requests.get(f"{API}/workspaces/{WORKSPACE_ID}/items?type=KQLDashboard", headers=HDR).json()["value"]
    import json as _json
    def _spotlight(dash):
        parts = _lro(requests.post(f"{API}/workspaces/{WORKSPACE_ID}/items/{dash['id']}/getDefinition", headers=HDR))["definition"]["parts"]
        idx = next(i for i, p in enumerate(parts) if p["path"].endswith("RealTimeDashboard.json"))
        j = _json.loads(_b64.b64decode(parts[idx]["payload"]).decode("utf-8"))
        ds_id = j["dataSources"][0]["id"]
        line_vo = next((t["visualOptions"] for t in j["tiles"] if t["visualType"] == "line"),
                       {"hideLegend": False, "selectedDataOnLoad": {"all": True, "limit": 10}})
        j["pages"]   = [p for p in j["pages"]   if p["id"] != PAGE_ID]
        j["queries"] = [q for q in j["queries"] if q["id"] not in (Q_TABLE, Q_LINE)]
        j["tiles"]   = [t for t in j["tiles"]   if t["id"] not in (T_TABLE, T_LINE)]
        j["pages"].insert(0, {"id": PAGE_ID, "name": "Today's Critical Tags"})
        ds = {"kind": "inline", "dataSourceId": ds_id}
        j["queries"] += [
            {"dataSource": ds, "text": table_text, "id": Q_TABLE, "usedVariables": []},
            {"dataSource": ds, "text": line_text,  "id": Q_LINE,  "usedVariables": []},
        ]
        j["tiles"] += [
            {"id": T_TABLE, "title": "Today's CRITICAL Tags - Actual vs Expected (Live)", "visualType": "table",
             "pageId": PAGE_ID, "layout": {"x": 0, "y": 0, "width": 24, "height": 13},
             "queryRef": {"kind": "query", "queryId": Q_TABLE},
             "visualOptions": {"colorRulesDisabled": True, "colorRules": [], "selectedDataOnLoad": {"all": True, "limit": 100}}},
            {"id": T_LINE, "title": "Today's CRITICAL Tags - Live Trend (60 min)", "visualType": "line",
             "pageId": PAGE_ID, "layout": {"x": 0, "y": 13, "width": 24, "height": 10},
             "queryRef": {"kind": "query", "queryId": Q_LINE}, "visualOptions": line_vo},
        ]
        parts[idx]["payload"] = _b64.b64encode(_json.dumps(j).encode("utf-8")).decode()
        parts[idx]["payloadType"] = "InlineBase64"
        _lro(requests.post(f"{API}/workspaces/{WORKSPACE_ID}/items/{dash['id']}/updateDefinition?updateMetadata=true",
                           headers=HDR, json={"definition": {"parts": parts}}))
    _targets = [i for i in items if i["displayName"] in ("Critical-Tag-Monitor", "pi-realtime-dashboard")]
    for _dash in _targets:
        _spotlight(_dash)
    print(f"Spotlight refreshed on {len(_targets)} dashboards: {len(tags)} critical tags.")
except Exception as e:
    print("Spotlight refresh skipped:", str(e)[:250])


In [ ]:
# OPTIONAL: nudge the Direct Lake semantic model to reframe (best-effort; safe to skip).
# Direct Lake normally picks up new Delta commits automatically on next query.
try:
    import requests, notebookutils
    WS = WORKSPACE_ID
    DS = "ac47a321-8bc2-4aa1-99f0-fc1a3ce06e42"   # semantic-main
    tok = notebookutils.credentials.getToken("pbi")
    r = requests.post(f"https://api.powerbi.com/v1.0/myorg/groups/{WS}/datasets/{DS}/refreshes",
                      headers={"Authorization": f"Bearer {tok}"}, json={"type": "full"})
    print("Model refresh trigger:", r.status_code)
except Exception as e:
    print("Refresh skipped (Direct Lake auto-reframes on query):", str(e)[:150])
